### Imports

In [1]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess

import math

### Data Processing

In [2]:
raw_ghcnd = pd.read_csv(r'../data/raw/GHCND_1980_to_2026.csv')
raw_ghcnd["DATE"] = pd.to_datetime(raw_ghcnd["DATE"])

raw_ghcnd.info()

<class 'pandas.DataFrame'>
RangeIndex: 17093 entries, 0 to 17092
Data columns (total 86 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   STATION          17093 non-null  str           
 1   NAME             17093 non-null  str           
 2   LATITUDE         17093 non-null  float64       
 3   LONGITUDE        17093 non-null  float64       
 4   ELEVATION        17093 non-null  float64       
 5   DATE             17093 non-null  datetime64[us]
 6   ACMH             5783 non-null   float64       
 7   ACMH_ATTRIBUTES  5783 non-null   str           
 8   ACSH             5783 non-null   float64       
 9   ACSH_ATTRIBUTES  5783 non-null   str           
 10  AWND             15629 non-null  float64       
 11  AWND_ATTRIBUTES  15629 non-null  str           
 12  FMTM             10180 non-null  float64       
 13  FMTM_ATTRIBUTES  10180 non-null  str           
 14  PGTM             10783 non-null  float64       
 

C:\Users\Moral\AppData\Local\Temp\ipykernel_31988\1601894922.py:1: DtypeWarning: Columns (7: ACMH_ATTRIBUTES, 9: ACSH_ATTRIBUTES, 13: FMTM_ATTRIBUTES, 19: PSUN_ATTRIBUTES, 25: TAVG_ATTRIBUTES, 31: TSUN_ATTRIBUTES, 33: WDF1_ATTRIBUTES, 39: WDFG_ATTRIBUTES, 41: WESD_ATTRIBUTES, 43: WSF1_ATTRIBUTES, 49: WSFG_ATTRIBUTES, 57: WT04_ATTRIBUTES, 59: WT05_ATTRIBUTES, 61: WT06_ATTRIBUTES, 63: WT07_ATTRIBUTES, 67: WT09_ATTRIBUTES, 69: WT10_ATTRIBUTES, 71: WT11_ATTRIBUTES, 73: WT13_ATTRIBUTES, 75: WT14_ATTRIBUTES, 77: WT15_ATTRIBUTES, 79: WT16_ATTRIBUTES, 81: WT18_ATTRIBUTES, 83: WT21_ATTRIBUTES, 85: WV03_ATTRIBUTES) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_ghcnd = pd.read_csv(r'../data/raw/GHCND_1980_to_2026.csv')


In [ ]:
def persistence_ghcnd_dataset(raw_ghcnd):
    updated_df = raw_ghcnd.set_index("DATE")

    TEMP_FEATURES = [
        "TMAX", "TMIN",
        "TMAX_lag1", "TMIN_lag1"
    ]

    # Create Lag Feature
    updated_df["TMAX_lag1"] = updated_df["TMAX"].shift(1)
    updated_df["TMIN_lag1"] = updated_df["TMIN"].shift(1)

    # Split Data
    updated_df = updated_df[6:]
    train_df = updated_df[(1980 <= updated_df.index.year) & (updated_df.index.year <= 2010)][TEMP_FEATURES]
    val_df = updated_df[(2011 <= updated_df.index.year) & (updated_df.index.year <= 2017)][TEMP_FEATURES]
    test_df = updated_df[(2018 <= updated_df.index.year) & (updated_df.index.year <= 2025)][TEMP_FEATURES]

    # Interpolate Data
    train_df.interpolate(method="time", inplace=True)
    val_df.interpolate(method="time", inplace=True)
    test_df.interpolate(method="time", inplace=True)

    # Apply Shifts and Take Out First 6 Value
    return train_df, val_df, test_df

def climatological_ghcnd_dataset(raw_ghcnd):
    updated_df = raw_ghcnd.set_index("DATE")

    # Add Values Fourier Calendar Values
    TEMP_FEATURES = [
        "TMAX", "TMIN",
    ]
    
    # Split Data
    train_df = updated_df[(1980 <= updated_df.index.year) & (updated_df.index.year <= 2010)][TEMP_FEATURES]
    val_df = updated_df[(2011 <= updated_df.index.year) & (updated_df.index.year <= 2017)][TEMP_FEATURES]
    test_df = updated_df[(2018 <= updated_df.index.year) & (updated_df.index.year <= 2025)][TEMP_FEATURES]

    # Interpolate Data
    train_df.interpolate(method="time", inplace=True)
    val_df.interpolate(method="time", inplace=True)
    test_df.interpolate(method="time", inplace=True)

    # Apply Shifts and Take Out First 6 Value
    return train_df, val_df, test_df

### Baseline Models

In [4]:
tabular_train_df, tabular_val_df, tabular_test_df = process_tabular_ghcnd(raw_ghcnd)
regular_train_df, _, _ = process_tabular_ghcnd(raw_ghcnd) # For Climatology as above removes the first 6 days of 1980

print(f"Train Size: {len(tabular_train_df)}")
print(f"Val Size: {len(tabular_val_df)}")
print(f"Test Size: {len(tabular_test_df)}")

Train Size: 11682
Val Size: 2557
Test Size: 2557


#### Metrics
- Metrics to Measure: (MAE, RMSE, Mean error, Baseline_skill)

In [5]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
def mean_error(true_y:np.ndarray, pred_y:np.ndarray):
    return (true_y - pred_y).mean(axis=0)

#### Persistence

In [6]:
class PersistenceBaseline():
    def __init__(self, train_df=None):
        pass

    def predict(self, input_df):
        output_df = input_df[["TMAX_lag1", "TMIN_lag1"]].rename(columns={"TMAX_lag1": "predicted_TMAX", "TMIN_lag1": "predicted_TMIN"})
        return output_df

In [7]:
persistence_model = PersistenceBaseline()

persistence_train_output = persistence_model.predict(tabular_train_df)
persistence_train_metrics = {
    "MAE":  mean_absolute_error(tabular_train_df[["TMAX", "TMIN"]].values, persistence_train_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(tabular_train_df[["TMAX", "TMIN"]].values, persistence_train_output.values, multioutput="raw_values"),
    "ME":   mean_error(tabular_train_df[["TMAX", "TMIN"]].values, persistence_train_output.values)
}

persistence_val_output = persistence_model.predict(tabular_val_df)
persistence_val_metrics = {
    "MAE":  mean_absolute_error(tabular_val_df[["TMAX", "TMIN"]].values, persistence_val_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(tabular_val_df[["TMAX", "TMIN"]].values, persistence_val_output.values, multioutput="raw_values"),
    "ME":   mean_error(tabular_val_df[["TMAX", "TMIN"]].values, persistence_val_output.values)
}

persistence_test_output = persistence_model.predict(tabular_test_df)
persistence_test_metrics = {
    "MAE":  mean_absolute_error(tabular_test_df[["TMAX", "TMIN"]].values, persistence_test_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(tabular_test_df[["TMAX", "TMIN"]].values, persistence_test_output.values, multioutput="raw_values"),
    "ME":   mean_error(tabular_test_df[["TMAX", "TMIN"]].values, persistence_test_output.values)
}

In [8]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- MAE: {persistence_train_metrics["MAE"]}")
print(f"- RMSE: {persistence_train_metrics["RMSE"]}")
print(f"- ME: {persistence_train_metrics["ME"]}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- MAE: {persistence_val_metrics["MAE"]}")
print(f"- RMSE: {persistence_val_metrics["RMSE"]}")
print(f"- ME: {persistence_val_metrics["ME"]}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- MAE: {persistence_test_metrics["MAE"]}")
print(f"- RMSE: {persistence_test_metrics["RMSE"]}")
print(f"- ME: {persistence_test_metrics["ME"]}\n")


Train Metrics: (TMAX/TMIN)
- MAE: [3.0431433  3.45771272]
- RMSE: [4.36107826 4.92185988]
- ME: [0.00188324 0.00196884]

Validation Metrics: (TMAX/TMIN)
- MAE: [2.93116934 3.09620649]
- RMSE: [4.18785513 4.4076142 ]
- ME: [0.00273758 0.00351975]

Test Metrics: (TMAX/TMIN)
- MAE: [2.90809542 3.05983575]
- RMSE: [4.13134948 4.3359523 ]
- ME: [-0.01016817 -0.01095033]



#### Climatological

In [9]:
class ClimatologicalBaseline():
    def __init__(self, train_df):
        tmp_df = train_df.copy(deep=True)
        tmp_df["DATE_OF_YEAR"] = train_df.index.strftime('%j').astype(int)
        self.climatological_model = tmp_df.groupby(by="DATE_OF_YEAR")[["TMAX", "TMIN"]].mean()

    def predict(self, input_df):
        tmp_df = input_df.copy(deep=True)
        tmp_df["DATE_OF_YEAR"] = tmp_df.index.strftime('%j').astype(int)
        return tmp_df[["DATE_OF_YEAR"]].join(self.climatological_model, on="DATE_OF_YEAR")[["TMAX", "TMIN"]].rename(columns={"TMAX": "predicted_TMAX", "TMIN": "predicted_TMIN"})

In [10]:
climatological_model = ClimatologicalBaseline(regular_train_df)

climatological_train_output = climatological_model.predict(regular_train_df)
climatological_train_metrics = {
    "MAE":  mean_absolute_error(regular_train_df[["TMAX", "TMIN"]].values, climatological_train_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(regular_train_df[["TMAX", "TMIN"]].values, climatological_train_output.values, multioutput="raw_values"),
    "ME":   mean_error(regular_train_df[["TMAX", "TMIN"]].values, climatological_train_output.values)
}

climatological_val_output = climatological_model.predict(tabular_val_df)
climatological_val_metrics = {
    "MAE":  mean_absolute_error(tabular_val_df[["TMAX", "TMIN"]].values, climatological_val_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(tabular_val_df[["TMAX", "TMIN"]].values, climatological_val_output.values, multioutput="raw_values"),
    "ME":   mean_error(tabular_val_df[["TMAX", "TMIN"]].values, climatological_val_output.values)
}

climatological_test_output = climatological_model.predict(tabular_test_df)
climatological_test_metrics = {
    "MAE":  mean_absolute_error(tabular_test_df[["TMAX", "TMIN"]].values, climatological_test_output.values, multioutput="raw_values"),
    "RMSE": root_mean_squared_error(tabular_test_df[["TMAX", "TMIN"]].values, climatological_test_output.values, multioutput="raw_values"),
    "ME":   mean_error(tabular_test_df[["TMAX", "TMIN"]].values, climatological_test_output.values)
}

In [11]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- MAE: {climatological_train_metrics["MAE"]}")
print(f"- RMSE: {climatological_train_metrics["RMSE"]}")
print(f"- ME: {climatological_train_metrics["ME"]}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- MAE: {climatological_val_metrics["MAE"]}")
print(f"- RMSE: {climatological_val_metrics["RMSE"]}")
print(f"- ME: {climatological_val_metrics["ME"]}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- MAE: {climatological_test_metrics["MAE"]}")
print(f"- RMSE: {climatological_test_metrics["RMSE"]}")
print(f"- ME: {climatological_test_metrics["ME"]}\n")

Train Metrics: (TMAX/TMIN)
- MAE: [3.88821547 4.7425183 ]
- RMSE: [5.31190617 6.43734815]
- ME: [-1.94635915e-17  5.93639540e-16]

Validation Metrics: (TMAX/TMIN)
- MAE: [4.13641474 5.12985464]
- RMSE: [5.45762778 6.75119176]
- ME: [1.05497953 2.63745553]

Test Metrics: (TMAX/TMIN)
- MAE: [4.29892247 5.41947311]
- RMSE: [5.45455483 6.83179831]
- ME: [2.0721872  3.55904333]



#### Comparing Climatology with Persistence

In [12]:
def baseline_skill(baseline_metrics, pred_metrics):
    return 1 - (pred_metrics['MAE']/baseline_metrics['MAE'])

In [13]:
print("Train Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_train_metrics, persistence_train_metrics)}\n")

print("Validation Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_val_metrics, persistence_val_metrics)}\n")

print("Test Metrics: (TMAX/TMIN)")
print(f"- Baseline: {baseline_skill(climatological_test_metrics, persistence_test_metrics)}\n")

Train Metrics: (TMAX/TMIN)
- Baseline: [0.21734191 0.2709121 ]

Validation Metrics: (TMAX/TMIN)
- Baseline: [0.29137441 0.39643387]

Test Metrics: (TMAX/TMIN)
- Baseline: [0.32352922 0.43539977]



### ML Models

#### Linear Regression

In [14]:
tabular_train_df.head()

,TMAX,TMIN,TMAX_lag1,TMIN_lag1,TMAX_lag2,TMIN_lag2,TMAX_lag3,TMIN_lag3,TMAX_lag4,TMIN_lag4,...,FOURIER_SIN_lag2,FOURIER_COS_lag2,FOURIER_SIN_lag3,FOURIER_COS_lag3,FOURIER_SIN_lag4,FOURIER_COS_lag4,FOURIER_SIN_lag5,FOURIER_COS_lag5,FOURIER_SIN_lag6,FOURIER_COS_lag6
DATE,,,,,,,,,,,,,,,,,,,,,
1980-01-07,66.0,37.0,54.0,36.0,60.0,44.0,66.0,51.0,67.0,35.0,...,0.068802,0.068802,0.051620,0.051620,0.034422,0.034422,0.017213,0.017213,0.000000,0.000000
1980-01-08,76.0,48.0,66.0,37.0,54.0,36.0,60.0,44.0,66.0,51.0,...,0.085965,0.085965,0.068802,0.068802,0.051620,0.051620,0.034422,0.034422,0.017213,0.017213
1980-01-09,76.0,62.0,76.0,48.0,66.0,37.0,54.0,36.0,60.0,44.0,...,0.103102,0.103102,0.085965,0.085965,0.068802,0.068802,0.051620,0.051620,0.034422,0.034422
1980-01-10,81.0,62.0,76.0,62.0,76.0,48.0,66.0,37.0,54.0,36.0,...,0.120208,0.120208,0.103102,0.103102,0.085965,0.085965,0.068802,0.068802,0.051620,0.051620
1980-01-11,80.0,66.0,81.0,62.0,76.0,62.0,76.0,48.0,66.0,37.0,...,0.137279,0.137279,0.120208,0.120208,0.103102,0.103102,0.085965,0.085965,0.068802,0.068802


In [15]:
tabular_train_df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 11682 entries, 1980-01-07 to 2011-12-31
Data columns (total 34 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   TMAX              11682 non-null  float64
 1   TMIN              11682 non-null  float64
 2   TMAX_lag1         11682 non-null  float64
 3   TMIN_lag1         11682 non-null  float64
 4   TMAX_lag2         11682 non-null  float64
 5   TMIN_lag2         11682 non-null  float64
 6   TMAX_lag3         11682 non-null  float64
 7   TMIN_lag3         11682 non-null  float64
 8   TMAX_lag4         11682 non-null  float64
 9   TMIN_lag4         11682 non-null  float64
 10  TMAX_lag5         11682 non-null  float64
 11  TMIN_lag5         11682 non-null  float64
 12  TMAX_lag6         11682 non-null  float64
 13  TMIN_lag6         11682 non-null  float64
 14  PRCP_lag1         11682 non-null  float64
 15  PRCP_lag2         11682 non-null  float64
 16  PRCP_lag3         11682 non-null  

In [ ]:
# Calculate temp deltas and ways to convert and revert them

def process_tabular_ghcnd(raw_ghcnd):
    updated_df = raw_ghcnd.set_index("DATE")
    updated_df["DATE_OF_YEAR"] = updated_df.index.map(
        lambda x: int(x.strftime('%j'))
    )

    # Add Values Fourier Calendar Values
    updated_df["FOURIER_SIN"] = updated_df.index.map(
        lambda x: math.sin(2*math.pi*(int(x.strftime('%j')) - 1)/365)
    )

    updated_df["FOURIER_COS"] = updated_df.index.map(
        lambda x: math.sin(2*math.pi*(int(x.strftime('%j')) - 1)/365)
    )

    # Normalize Temperature
    train_mean = 0
    train_var = 0


    TEMP_FEATURES = [
        "TMAX", "TMIN",
        "TMAX_lag1", "TMIN_lag1", 
        "TMAX_lag2", "TMIN_lag2", 
        "TMAX_lag3", "TMIN_lag3", 
        "TMAX_lag4", "TMIN_lag4", 
        "TMAX_lag5", "TMIN_lag5",
        "TMAX_lag6", "TMIN_lag6",
    ]
    PRCP_FEATURES = [
        "PRCP_lag1",
        "PRCP_lag2",
        "PRCP_lag3",
        "PRCP_lag4",
        "PRCP_lag5",
        "PRCP_lag6",
    ]
    FOURIER_FEATURES = [
        "FOURIER_SIN", "FOURIER_COS",
        "FOURIER_SIN_lag1", "FOURIER_COS_lag1",
        "FOURIER_SIN_lag2", "FOURIER_COS_lag2",
        "FOURIER_SIN_lag3", "FOURIER_COS_lag3",
        "FOURIER_SIN_lag4", "FOURIER_COS_lag4",
        "FOURIER_SIN_lag5", "FOURIER_COS_lag5",
        "FOURIER_SIN_lag6", "FOURIER_COS_lag6",
    ]

    updated_df["TMAX_lag1"] = updated_df["TMAX"].shift(1)
    updated_df["TMIN_lag1"] = updated_df["TMIN"].shift(1)
    updated_df["PRCP_lag1"] = updated_df["PRCP"].shift(1)
    updated_df["FOURIER_SIN_lag1"] = updated_df["FOURIER_SIN"].shift(1)
    updated_df["FOURIER_COS_lag1"] = updated_df["FOURIER_COS"].shift(1)

    updated_df["TMAX_lag2"] = updated_df["TMAX"].shift(2)
    updated_df["TMIN_lag2"] = updated_df["TMIN"].shift(2)
    updated_df["PRCP_lag2"] = updated_df["PRCP"].shift(2)
    updated_df["FOURIER_SIN_lag2"] = updated_df["FOURIER_SIN"].shift(2)
    updated_df["FOURIER_COS_lag2"] = updated_df["FOURIER_COS"].shift(2)

    updated_df["TMAX_lag3"] = updated_df["TMAX"].shift(3)
    updated_df["TMIN_lag3"] = updated_df["TMIN"].shift(3)
    updated_df["PRCP_lag3"] = updated_df["PRCP"].shift(3)
    updated_df["FOURIER_SIN_lag3"] = updated_df["FOURIER_SIN"].shift(3)
    updated_df["FOURIER_COS_lag3"] = updated_df["FOURIER_COS"].shift(3)

    updated_df["TMAX_lag4"] = updated_df["TMAX"].shift(4)
    updated_df["TMIN_lag4"] = updated_df["TMIN"].shift(4)
    updated_df["PRCP_lag4"] = updated_df["PRCP"].shift(4)
    updated_df["FOURIER_SIN_lag4"] = updated_df["FOURIER_SIN"].shift(4)
    updated_df["FOURIER_COS_lag4"] = updated_df["FOURIER_COS"].shift(4)

    updated_df["TMAX_lag5"] = updated_df["TMAX"].shift(5)
    updated_df["TMIN_lag5"] = updated_df["TMIN"].shift(5)
    updated_df["PRCP_lag5"] = updated_df["PRCP"].shift(5)
    updated_df["FOURIER_SIN_lag5"] = updated_df["FOURIER_SIN"].shift(5)
    updated_df["FOURIER_COS_lag5"] = updated_df["FOURIER_COS"].shift(5)

    updated_df["TMAX_lag6"] = updated_df["TMAX"].shift(6)
    updated_df["TMIN_lag6"] = updated_df["TMIN"].shift(6)
    updated_df["PRCP_lag6"] = updated_df["PRCP"].shift(6)
    updated_df["FOURIER_SIN_lag6"] = updated_df["FOURIER_SIN"].shift(6)
    updated_df["FOURIER_COS_lag6"] = updated_df["FOURIER_COS"].shift(6)
    
    # Split Data
    ALL_FEATURES = TEMP_FEATURES + PRCP_FEATURES + FOURIER_FEATURES
    updated_df = updated_df[6:]
    train_df = updated_df[(1980 <= updated_df.index.year) & (updated_df.index.year <= 2011)][ALL_FEATURES]
    val_df = updated_df[(2012 <= updated_df.index.year) & (updated_df.index.year <= 2018)][ALL_FEATURES]
    test_df = updated_df[(2019 <= updated_df.index.year) & (updated_df.index.year <= 2025)][ALL_FEATURES]

    # Interpolate Data
    train_df.interpolate(method="time", inplace=True)
    val_df.interpolate(method="time", inplace=True)
    test_df.interpolate(method="time", inplace=True)

    # Apply Shifts and Take Out First 6 Value
    return train_df, val_df, test_df


In [21]:
(regular_train_df[["TMAX", "TMIN"]] - regular_train_df[["TMAX", "TMIN"]].mean())/regular_train_df[["TMAX", "TMIN"]].std()

,TMAX,TMIN
DATE,,
1980-01-07,-1.737051,-2.469524
1980-01-08,-0.653308,-1.479554
1980-01-09,-0.653308,-0.219593
1980-01-10,-0.111436,-0.219593
1980-01-11,-0.219810,0.140396
...,...,...
2011-12-27,-0.870056,-0.219593
2011-12-28,-1.953800,-1.299560
2011-12-29,-1.520302,-1.569552


#### XGBoost

#### RNN